In [1]:
import pandas as pd
import numpy as np
import warnings
from scipy.stats import spearmanr
import optuna
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

In [2]:
df = pd.read_parquet('BTC_1h_logRV_ML_dataset.parquet')
df.head()

,log_rv_lag_1h,log_rv_lag_2h,log_rv_lag_3h,log_rv_lag_6h,log_rv_lag_12h,log_rv_lag_24h,log_rv_lag_48h,log_rv_lag_72h,log_rv_lag_168h,log_rv_mean_3h,...,is_us_session,is_eu_us_overlap,is_low_vol_168h,is_high_vol_168h,log_rv_lag1_x_us_session,log_rv_lag1_x_weekend,log_volume_lag1_x_us_session,abs_ret_lag1_x_us_session,target_log_rv,target_rv
datetime,,,,,,,,,,,,,,,,,,,,,
2020-04-24 10:00:00+00:00,-11.037855,-10.484218,-10.835839,-10.888887,-10.680269,-11.994565,-12.063372,-10.541077,-11.278820,-10.785971,...,0,0,0,0,-0.000000,-0.0,0.000000,0.000000,-10.399756,0.000030
2020-04-24 11:00:00+00:00,-10.399756,-11.037855,-10.484218,-11.590795,-9.354094,-11.090701,-11.917092,-9.388464,-11.179881,-10.640610,...,0,0,0,0,-0.000000,-0.0,0.000000,0.000000,-9.136467,0.000108
2020-04-24 12:00:00+00:00,-9.136467,-10.399756,-11.037855,-12.173192,-10.142192,-10.344540,-11.376537,-10.698591,-11.013714,-10.191359,...,0,0,0,1,-0.000000,-0.0,0.000000,0.000000,-9.878886,0.000051
2020-04-24 13:00:00+00:00,-9.878886,-9.136467,-10.399756,-10.835839,-10.731020,-9.506733,-9.125376,-10.832082,-9.459211,-9.805036,...,1,1,0,0,-9.878886,-0.0,6.351896,0.001266,-11.220871,0.000013
2020-04-24 14:00:00+00:00,-11.220871,-9.878886,-9.136467,-10.484218,-11.638972,-6.899771,-9.461127,-10.254837,-10.610571,-10.078742,...,1,1,0,0,-11.220871,-0.0,5.967658,0.003403,-9.154555,0.000106


In [3]:
def make_walk_forward_splits(
    data,
    train_years=2,
    val_months=1,
    test_months=2,
    step_months=2,
    use_val=True
):
    start = data.index.min() + pd.DateOffset(years=train_years)
    end = data.index.max()

    splits = []
    split_start = start

    while True:
        train_start = split_start - pd.DateOffset(years=train_years)
        train_end = split_start

        if use_val:
            val_start = train_end
            val_end = val_start + pd.DateOffset(months=val_months)
            test_start = val_end
            test_end = test_start + pd.DateOffset(months=test_months)
        else:
            val_start = None
            val_end = None
            test_start = train_end
            test_end = test_start + pd.DateOffset(months=test_months)

        if test_end > end:
            break

        train_idx = (data.index >= train_start) & (data.index < train_end)
        test_idx = (data.index >= test_start) & (data.index < test_end)

        if use_val:
            val_idx = (data.index >= val_start) & (data.index < val_end)
        else:
            val_idx = None

        splits.append({
            "train_idx": train_idx,
            "val_idx": val_idx,
            "test_idx": test_idx,
            "train_start": train_start,
            "train_end": train_end,
            "val_start": val_start,
            "val_end": val_end,
            "test_start": test_start,
            "test_end": test_end
        })

        split_start = split_start + pd.DateOffset(months=step_months)

    return splits

In [4]:
def qlike(y_true_rv, y_pred_rv, eps=1e-12):
    y_true_rv = np.maximum(y_true_rv, eps)
    y_pred_rv = np.maximum(y_pred_rv, eps)

    return np.mean(
        y_true_rv / y_pred_rv - np.log(y_true_rv / y_pred_rv) - 1
    )


def regression_metrics(y_true_log, y_pred_log, eps=1e-12):
    y_true_log = np.asarray(y_true_log)
    y_pred_log = np.asarray(y_pred_log)

    y_true_rv = np.exp(y_true_log) - eps
    y_pred_rv = np.exp(y_pred_log) - eps

    return {
        "mse_log": np.mean((y_true_log - y_pred_log) ** 2),
        "mae_log": np.mean(np.abs(y_true_log - y_pred_log)),
        "mse_rv": np.mean((y_true_rv - y_pred_rv) ** 2),
        "mae_rv": np.mean(np.abs(y_true_rv - y_pred_rv)),
        "qlike": qlike(y_true_rv, y_pred_rv),
        "pearson": np.corrcoef(y_true_log, y_pred_log)[0, 1],
        "spearman": spearmanr(y_true_log, y_pred_log).correlation
    }

In [5]:
def run_walk_forward_with_val(
    model_fn,
    dataset,
    feature_cols,
    target_col="target_log_rv",
    output_path="predictions_walk_forward_val.parquet"
):
    splits = make_walk_forward_splits(
        dataset,
        train_years=2,
        val_months=1,
        test_months=2,
        step_months=2,
        use_val=True
    )

    preds = []
    metrics = []

    for i, split in enumerate(splits):
        train = dataset.loc[split["train_idx"]]
        val = dataset.loc[split["val_idx"]]
        test = dataset.loc[split["test_idx"]]

        model = model_fn()

        model.fit(
            train[feature_cols],
            train[target_col],
            eval_set=[(val[feature_cols], val[target_col])]
        )

        y_pred = model.predict(test[feature_cols])
        y_true = test[target_col].values

        fold_preds = pd.DataFrame({
            "datetime": test.index,
            "fold": i,
            "y_true_log_rv": y_true,
            "y_pred_log_rv": y_pred,
            "y_true_rv": np.exp(y_true) - 1e-12,
            "y_pred_rv": np.exp(y_pred) - 1e-12
        })

        preds.append(fold_preds)

        fold_metrics = regression_metrics(y_true, y_pred)
        fold_metrics["fold"] = i
        fold_metrics["test_start"] = split["test_start"]
        fold_metrics["test_end"] = split["test_end"]
        metrics.append(fold_metrics)

    preds = pd.concat(preds).set_index("datetime")
    metrics = pd.DataFrame(metrics)

    preds.to_parquet(output_path)

    return preds, metrics, metrics.drop(columns=["fold", "test_start", "test_end"]).mean()

In [7]:
optuna.logging.set_verbosity(optuna.logging.WARNING)


def qlike_loss(y_true, y_pred):
    rv_true = np.exp(y_true)
    rv_pred = np.exp(y_pred)
    ratio = rv_true / np.maximum(rv_pred, 1e-12)
    return np.mean(ratio - np.log(ratio) - 1)


class OptunaRidge:
    def __init__(self, n_trials=15):
        self.n_trials = n_trials

    def fit(self, X, y, eval_set=None):
        X_val, y_val = eval_set[0]

        def objective(trial):
            alpha = trial.suggest_float("alpha", 1e-3, 10.0, log=True)

            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("model", Ridge(alpha=alpha))
            ])

            pipe.fit(X, y)

            pred = pipe.predict(X_val)

            return qlike_loss(y_val, pred)

        study = optuna.create_study(direction="minimize")
        study.optimize(objective, n_trials=self.n_trials, show_progress_bar=False)

        self.best_params_ = study.best_params

        self.model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=study.best_params["alpha"]))
        ])

        self.model.fit(
            pd.concat([X, X_val]),
            pd.concat([y, y_val])
        )

        return self

    def predict(self, X):
        return self.model.predict(X)


df = pd.read_parquet("BTC_1h_logRV_ML_dataset.parquet").sort_index()

y = df["target_log_rv"]
rv = np.exp(y)

df["target_log_rv_1h"] = y.shift(-1)

df["target_log_rv_6h"] = np.log(
    rv.shift(-1).rolling(6).mean().shift(-5)
)

df["target_log_rv_24h"] = np.log(
    rv.shift(-1).rolling(24).mean().shift(-23)
)

df["target_log_rv_168h"] = np.log(
    rv.shift(-1).rolling(168).mean().shift(-167)
)

targets = [
    "target_log_rv_1h",
    "target_log_rv_6h",
    "target_log_rv_24h",
    "target_log_rv_168h",
]

drop_cols = [
    "target_log_rv",
    "target_rv",
    "target_log_rv_1h",
    "target_log_rv_6h",
    "target_log_rv_24h",
    "target_log_rv_168h",
]

feature_cols = [
    c for c in df.columns
    if c not in drop_cols
]

all_preds = []
all_metrics = []

for target in targets:

    data = df[
        feature_cols + [target]
    ].replace([np.inf, -np.inf], np.nan).dropna()

    preds, metrics, avg_metrics = run_walk_forward_with_val(
        model_fn=lambda: OptunaRidge(n_trials=15),
        dataset=data,
        feature_cols=feature_cols,
        target_col=target,
        output_path=f"Ridge_Full_{target}.parquet"
    )

    preds = preds.reset_index()
    preds["model"] = "Ridge_Full"
    preds["target"] = target

    all_preds.append(preds)

    metrics["model"] = "Ridge_Full"
    metrics["target"] = target

    all_metrics.append(metrics)

ridge_full_preds = pd.concat(all_preds)
ridge_full_metrics = pd.concat(all_metrics).reset_index(drop=True)

ridge_full_preds.to_parquet(
    "ridge_full.parquet",
    index=False
)

ridge_full_metrics.to_parquet(
    "ridge_full_metrics.parquet",
    index=False
)